# Phase 6 - full grid (Colab)

10 seeds x {linear, mlp} x 10 conditions x 3 datasets = **600 cells**.
~3-4 h on Colab CPU (feature matrices are cached per condition). Needs the
Phase 3 embeddings in Drive.

`--out` writes straight to a Drive parquet, flushed after every cell, so a
disconnect loses nothing - just re-run the last cell to resume. The 90 pilot
cells (linear, seeds 0-2) are reused as a head start (bit-identical to the
refactored path - verified).

In [ ]:
!git clone https://github.com/ryanteachman/sbert-head-ablation.git
%cd sbert-head-ablation
!pip install -q pyarrow pyyaml scikit-learn
import torch; print('torch', torch.__version__, '| cuda', torch.cuda.is_available())

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
DRIVE = '/content/drive/MyDrive/sbert-head-ablation'
OUT = f'{DRIVE}/results/runs.parquet'
import os, shutil, json
os.makedirs(f'{DRIVE}/results', exist_ok=True)
!mkdir -p embeddings && rsync -a "{DRIVE}/embeddings/" embeddings/
assert len(json.load(open('embeddings/meta.json'))['splits']) == 11, 'run embed_colab.ipynb first'
# head start from the pilot (linear, seeds 0-2) if the full run hasn't begun
if not os.path.exists(OUT) and os.path.exists(f'{DRIVE}/results/pilot_runs.parquet'):
    shutil.copy(f'{DRIVE}/results/pilot_runs.parquet', OUT); print('seeded runs.parquet from pilot')
import pandas as pd
print('starting from', len(pd.read_parquet(OUT)) if os.path.exists(OUT) else 0, 'cells')

## 1. Quick check (2 cells)
Confirms the pipeline runs on the real embeddings before the long haul.

In [ ]:
!python src/run_grid.py --embed-dir embeddings --out /content/_check.parquet \
  --datasets paws --conditions C3 --heads linear,mlp --seeds 7 --limit 2
import pandas as pd; print(pd.read_parquet('/content/_check.parquet')[['condition','head','seed','test_acc','wall_s']].to_string())

## 2. Full grid
Resumable. If Colab disconnects, just re-run this cell - it skips finished
cells. Progress prints per cell (`condition head sN  acc=... [Nep Ns]`).

In [ ]:
!python src/run_grid.py --embed-dir embeddings --out "{OUT}"

In [ ]:
import pandas as pd
df = pd.read_parquet(OUT)
print(f'{len(df)} / 600 cells\n')
for h in ['linear', 'mlp']:
    sub = df[df.head == h]
    if sub.empty: continue
    print(f'--- {h} : mean test_acc ---')
    print(sub.pivot_table(index='dataset', columns='condition', values='test_acc', aggfunc='mean').round(4).to_string())
    print()
ceiling = int(((df.epochs_trained == df.epochs_trained.max()) & (~df.early_stopped)).sum())
print('epoch-ceiling hits:', ceiling)
if len(df) == 600:
    from google.colab import files; files.download(OUT)

## Done
Download `runs.parquet` (also in Drive under `results/`), drop it in `results/`
locally, commit. Phase 7 analysis runs locally off that file.